# Smart-Turn — load exported dataset

Entry-point notebook for the smart-turn workflow. It picks up where
[`wk exports adapt smart-turn`](https://github.com/wavekat/wavekat-cli) leaves
off: a directory of Parquet shards plus a `wavekat_export_meta.json`
provenance file.

What this notebook does:

1. Load `train` / `validation` / `test` Parquet shards into a
   `DatasetDict` with the HuggingFace `Audio` feature attached.
2. Sanity-check the splits — counts, label balance, clip-duration
   distribution.
3. Audition a few clips from each class so you know the audio is
   actually decoding.
4. Show the export provenance so a future reader can trace this
   notebook back to the snapshot that produced it.

Downstream notebooks (training, eval) consume the same `ds` shape this
notebook produces — keep the loader cell stable.

## Configure

Point `EXPORT_DIR` at whatever directory you passed to `wk exports adapt
smart-turn --out`. The default below assumes the example layout from the
wavekat-cli README (`./datasets/smart-turn-zh`).

In [ ]:
from pathlib import Path

EXPORT_DIR = Path("../../datasets/smart-turn-zh").resolve()
TARGET_SR = 16_000  # smart-turn operates at 16 kHz; the adapter writes at 16 kHz too.

assert EXPORT_DIR.exists(), (
    f"{EXPORT_DIR} does not exist. Run `wk exports adapt smart-turn --out {EXPORT_DIR}` first, "
    f"or edit EXPORT_DIR above."
)
print("export dir:", EXPORT_DIR)
print("contents :", sorted(p.name for p in EXPORT_DIR.iterdir()))

## Load

`validation.parquet` and `test.parquet` are optional — the adapter only
writes a shard for splits that actually contain rows. We pick up
whatever is on disk so the notebook works for both full and train-only
exports.

In [ ]:
from datasets import load_dataset, Audio

data_files = {
    split: str(EXPORT_DIR / f"{split}.parquet")
    for split in ("train", "validation", "test")
    if (EXPORT_DIR / f"{split}.parquet").exists()
}
assert "train" in data_files, "train.parquet is missing — adapt step did not produce it."

ds = load_dataset("parquet", data_files=data_files)
ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))
ds

## Split sizes and label balance

`endpoint_bool` is the binary smart-turn target — 1 = `end_of_turn`,
0 = `continuation`. A wildly skewed balance here is your earliest
warning that the export filter pulled the wrong slice.

In [ ]:
import pandas as pd

rows = []
for split, dset in ds.items():
    labels = dset["endpoint_bool"]
    pos = sum(labels)
    neg = len(labels) - pos
    rows.append(
        {
            "split": split,
            "n": len(labels),
            "end_of_turn": pos,
            "continuation": neg,
            "% end_of_turn": round(100 * pos / max(len(labels), 1), 1),
        }
    )
pd.DataFrame(rows).set_index("split")

## Clip duration distribution

Smart-turn cares about the tail — clips much shorter or longer than the
rest will train poorly. The adapter passes `clip_duration_sec` through
from the canonical snapshot, so this is the source of truth (no decode
needed).

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for split, dset in ds.items():
    durations = dset["clip_duration_sec"]
    ax.hist(durations, bins=40, alpha=0.5, label=f"{split} (n={len(durations)})")
ax.set_xlabel("clip_duration_sec")
ax.set_ylabel("count")
ax.set_title("Clip duration by split")
ax.legend()
plt.show()

pd.DataFrame(
    {split: pd.Series(dset["clip_duration_sec"]).describe() for split, dset in ds.items()}
)

## Audition

Listen to a sample of each class from `train`. If the playback widget
below is silent, the audio bytes round-tripped but didn't decode — most
likely a sample-rate mismatch between the export and `TARGET_SR`.

In [ ]:
from IPython.display import Audio as AudioWidget, display
import random

random.seed(0)
train = ds["train"]
by_label = {0: [], 1: []}
for i, row in enumerate(train):
    by_label[row["endpoint_bool"]].append(i)
    if all(len(v) >= 8 for v in by_label.values()):
        break

for label, label_name in [(1, "end_of_turn"), (0, "continuation")]:
    sample_idx = random.sample(by_label[label], k=min(2, len(by_label[label])))
    for idx in sample_idx:
        row = train[idx]
        print(
            f"[{label_name}] {row['annotation_id']}  duration={row['clip_duration_sec']:.2f}s  "
            f"path={row['audio']['path']}"
        )
        display(AudioWidget(row["audio"]["array"], rate=row["audio"]["sampling_rate"]))

## Provenance

`wavekat_export_meta.json` is the breadcrumb that ties this notebook
back to the snapshot directory and `wk` invocation that produced it.
Save its contents into any downstream artifact (model card, eval
report) so the dataset is traceable.

In [ ]:
import json

meta_path = EXPORT_DIR / "wavekat_export_meta.json"
meta = json.loads(meta_path.read_text()) if meta_path.exists() else None
meta

## Next

`ds` is ready for training. The expected downstream notebooks:

- `02_train_smart_turn.ipynb` — fine-tune the smart-turn model on
  `ds["train"]`, validate on `ds["validation"]`.
- `03_eval_smart_turn.ipynb` — held-out metrics on `ds["test"]`.

Both notebooks should reload the dataset using the same `EXPORT_DIR` /
`load_dataset` block above, not a private cache, so a new export is
picked up automatically.